# 02 · Curvas y valorización por flujos

Muestra la capa (ii) (`src/optimum/curves.py`, `valuation.py`): curvas base, coherencia de forwards, calibración de spreads (V0 = monto de mercado al 31-12-2025), resultado al horizonte sin shock y sensibilidad del patrimonio por **revalorización completa** (sin duración ni DV01).

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from optimum.cleaning import load_processed
from optimum.reporting.figures import COLORS, new_figure, save_fig
from optimum.reporting.tables import save_table

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## Curvas spot al 31-12-2025

In [ ]:
from optimum.curves import ZeroCurve, implied_forward

hist = load_processed("market_history")
base = hist.loc["2025-12-31"]
curves = {c: ZeroCurve.from_market_row(base, c) for c in ["PEN", "USD"]}
t = np.linspace(0, 20, 201)
fig, ax = new_figure()
for c, cv in curves.items():
    ax.plot(t, 100 * cv.zero(t), color=COLORS[c], label=c)
    ax.scatter(cv.tenors, 100 * cv.rates, color=COLORS[c], s=12)
ax.set(xlabel="Años", ylabel="Tasa cero (%)", title="Curvas spot al 31-12-2025")
ax.legend()
save_fig(fig, "curvas_spot_base")

## Forwards implícitas vs. forwards del Excel

In [ ]:
from optimum.curves import NODE_YEARS, SEGMENT_BOUNDS

rows = []
for c, cv in curves.items():
    for s, (a, b) in SEGMENT_BOUNDS.items():
        t1, t2 = max(NODE_YEARS[a], 1e-9), NODE_YEARS[b]
        rows.append({"ccy": c, "seg": s, "implícita": implied_forward(cv, t1, t2), "excel": base[f"{c}_FWD_{s}"]})
fwd = pd.DataFrame(rows).assign(dif_pb=lambda d: 1e4 * (d["implícita"] - d["excel"]))
print(f"Máxima diferencia: {fwd['dif_pb'].abs().max():.4f} pb")
fwd.round(5)

## Calibración de spreads

Para cada instrumento con flujos se resuelve el spread $s$ tal que $\sum_t CF_t (1 + z(t) + s)^{-t} \cdot FX_0 = $ `market_value_pen`. `Calibration_Spreads` del Excel es solo una referencia.

In [ ]:
from optimum import valuation as val
from optimum.io.json_contract import load_input

doc = load_input()
spreads = val.calibrate_spreads(doc)
v0 = val.value_at_t0(doc, spreads)
inst = pd.DataFrame(doc["instruments"]).set_index("id")
calib = pd.DataFrame(
    {
        "tipo": inst["instrument_type"],
        "moneda": inst["currency"],
        "spread Excel (pb)": 1e4 * pd.Series(doc["calibrationSpreads"]),
        "spread calibrado (pb)": 1e4 * pd.Series(spreads),
        "V0 (S/ mm)": v0,
        "mercado (S/ mm)": inst["market_value_pen"],
    }
)
calib["dif. spread (pb)"] = calib["spread calibrado (pb)"] - calib["spread Excel (pb)"]
assert np.allclose(calib["V0 (S/ mm)"], calib["mercado (S/ mm)"], atol=1e-6)
save_table(
    calib.drop(columns=["mercado (S/ mm)"]),
    "spreads_calibrados",
    caption="Spreads de calibración (V0 = valor de mercado al 31-12-2025).",
    label="tab:spreads",
    fmt="{:,.2f}",
)
calib.round(3)

La diferencia con el Excel es menor a 0.5 pb. Su origen no se investigó (podría ser una convención de conteo de días o de descuento distinta en la hoja). No afecta los resultados: el modelo usa el spread recalibrado, que reproduce V0 exactamente.

## Resultado al horizonte sin shock (carry)

$r_k = \dfrac{V_k(t_H) + CF_k(t_0, t_H]}{V_k(t_0)} - 1$ con $\Delta = 0$ (formulación §7.4). Para los pasivos, $r_j$ es su costo.

In [ ]:
h0 = val.horizon_value_and_flows(doc, spreads, {})
carry = h0.assign(v0=v0, lado=inst["side"])
carry["r (%)"] = 100 * ((carry["value_h"] + carry["flows"]) / carry["v0"] - 1)
sign = np.where(carry["lado"] == "ASSET", 1, -1)
pn0 = (sign * carry["v0"]).sum()
pn_h = (sign * (carry["value_h"] + carry["flows"])).sum()
print(f"PN0 = {pn0:,.2f}  →  PN(tH) sin shock = {pn_h:,.2f}  (ΔPN = {pn_h - pn0:+,.2f} S/ mm)")
save_table(
    carry[["lado", "v0", "value_h", "flows", "r (%)"]].rename(
        columns={"v0": "V0", "value_h": "V(tH)", "flows": "Flujos año"}
    ),
    "carry_base",
    caption="Resultado a un año sin shock por instrumento (S/ mm).",
    label="tab:carry",
)
carry.round(3)

## Sensibilidad del patrimonio por revalorización

Shocks instantáneos en $t_0$ aplicados con `value_at_t0`: cada instrumento se revaloriza con la curva shockeada (forwards implícitas recalculadas y cupones flotantes reproyectados). $\Delta PN = \Delta A - \Delta L$.

In [ ]:
shocks = {
    "PEN +100 pb": {f"PEN_SPOT_{n}": 0.01 for n in NODE_YEARS},
    "USD +100 pb": {f"USD_SPOT_{n}": 0.01 for n in NODE_YEARS},
    "FX +10 %": {"FX_PENUSD": np.log(1.10)},
    "Equity −10 %": {"EQUITY_USD": np.log(0.90)},
}
dv = pd.DataFrame({k: val.value_at_t0(doc, spreads, d) - v0 for k, d in shocks.items()})
is_asset = inst["side"] == "ASSET"
impact = pd.DataFrame(
    {"ΔA": dv[is_asset].sum(), "ΔL": dv[~is_asset].sum()}
).assign(**{"ΔPN": lambda d: d["ΔA"] - d["ΔL"]})
fig, ax = new_figure(aspect=0.45)
x = np.arange(len(impact))
ax.bar(x - 0.27, impact["ΔA"], 0.27, color=COLORS["ASSET"], label="ΔActivos")
ax.bar(x, -impact["ΔL"], 0.27, color=COLORS["LIABILITY"], label="−ΔPasivos")
ax.bar(x + 0.27, impact["ΔPN"], 0.27, color=COLORS["EQUITY"], label="ΔPN")
ax.axhline(0, color="black", lw=0.6)
ax.set(xticks=x, xticklabels=impact.index, ylabel="S/ mm", title="Sensibilidad del patrimonio (revalorización en t0)")
ax.legend()
save_fig(fig, "sensibilidad_pn_shocks")
impact.round(2)

In [ ]:
dv.round(2)